# Topic 3: Catalyst Optimizer & Explain Plans

> **Phase 3 → Week 5 → Topic 3**

---

## What is the Catalyst Optimizer?

Catalyst is Spark's query optimizer — it sits between your code and the physical execution. It transforms your logical query into an optimal physical execution plan.

```
Your Code (SQL or DataFrame API)
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│                    CATALYST OPTIMIZER                        │
│                                                             │
│  1. Unresolved Logical Plan  ──→  Analysis                  │
│     (parse SQL/DataFrame)         (resolve names & types)   │
│                                          │                  │
│  2. Resolved Logical Plan   ◄────────────┘                  │
│                                          │                  │
│  3. Optimized Logical Plan  ◄─── Optimization Rules:        │
│                                  • Predicate Pushdown        │
│                                  • Column Pruning            │
│                                  • Constant Folding          │
│                                  • Join Reordering           │
│                                          │                  │
│  4. Physical Plans          ◄────────────┘                  │
│     (multiple candidates)        Cost Model selects best    │
│                                          │                  │
│  5. Selected Physical Plan  ◄────────────┘                  │
│     + Tungsten Code Gen                                     │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
    Execution (Jobs → Stages → Tasks)
```

---

## Key Optimization Rules

### 1. Predicate Pushdown
Move filters as early as possible — ideally down to the data source level.
```
Before:  Read all data → join → filter
After:   Filter at source → join (fewer rows to join = faster)
```

### 2. Column Pruning
Only read the columns actually needed.
```
SELECT name, salary FROM employees
→ Parquet reader only reads 2 columns instead of all 20
```

### 3. Constant Folding
Evaluate constant expressions at compile time.
```
WHERE price > 100 * 10  →  WHERE price > 1000  (pre-computed)
```

### 4. AQE — Adaptive Query Execution (Spark 3.0+)
Re-optimizes the plan at runtime based on actual data statistics.
- Auto-coalesces shuffle partitions (avoids too many tiny partitions)
- Converts sort-merge join to broadcast join if runtime sizes allow
- Fixes skewed joins at runtime

---

## Interview Cheat Sheet

**Q: What is the Catalyst Optimizer?**
> Catalyst is Spark's query optimization framework. It transforms the DataFrame/SQL logical plan through a series of rule-based and cost-based optimizations — predicate pushdown (filter early), column pruning (read only needed columns), constant folding, and join reordering — before generating the physical execution plan.

**Q: What is predicate pushdown and why does it matter?**
> Predicate pushdown moves filter conditions as close to the data source as possible. For Parquet files, the filter is pushed into the Parquet reader itself — rows that don't match are never read into memory. For a 100GB table where you filter 99% of rows, this reduces I/O by 99% — the biggest performance win possible.

**Q: What is AQE (Adaptive Query Execution)?**
> AQE (introduced in Spark 3.0) re-optimizes queries at runtime using actual data statistics collected during execution. It can auto-coalesce shuffle partitions (e.g., 200 partitions down to 5 if the data is small), switch from sort-merge join to broadcast join when the runtime size is small enough, and automatically fix data skew in joins.

**Q: How do you read a Spark explain() output?**
> Read bottom-up. The bottom node is the data source (scan). Moving up, you see transformations applied in order. Key nodes: `FileScan` (data read with pushed filters), `Filter` (applied after read), `Project` (column selection), `HashAggregate` (groupBy), `Exchange` (shuffle), `SortMergeJoin` or `BroadcastHashJoin`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Week5 - Catalyst") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("/workspace/week4/data/orders.csv",    header=True, inferSchema=True)
products  = spark.read.csv("/workspace/week4/data/products.csv",  header=True, inferSchema=True)
print("Ready")

## Part 1: Reading explain() Output

In [ ]:
# Simple explain — physical plan only
query = orders.filter(F.col("amount") > 500) \
              .groupBy("customer_id") \
              .agg(F.sum("amount").alias("total"))

print("=== explain() — Physical Plan (read bottom-up) ===")
query.explain()
print("""
Reading the plan (bottom to top):
  - FileScan: reads the CSV file
  - Filter: applies amount > 500 (may be pushed down)
  - HashAggregate: partial aggregation per partition
  - Exchange: shuffle by customer_id (groups data together)
  - HashAggregate: final aggregation after shuffle
""")

In [ ]:
# explain(True) or explain("extended") — shows all 4 plans
print("=== explain('extended') — all 4 plan stages ===")
query.explain("extended")

In [ ]:
# explain("formatted") — cleaner tree format (Spark 3.0+)
print("=== explain('formatted') — clean tree view ===")
query.explain("formatted")

## Part 2: Predicate Pushdown in Action

In [ ]:
# Write data as Parquet to see pushdown
orders.write.mode("overwrite").parquet("/tmp/orders_parquet")
products.write.mode("overwrite").parquet("/tmp/products_parquet")

orders_pq   = spark.read.parquet("/tmp/orders_parquet")
products_pq = spark.read.parquet("/tmp/products_parquet")

# Filter AFTER read
df_late_filter = orders_pq.filter(F.col("amount") > 500)
print("Filter after read — check plan for PushedFilters:")
df_late_filter.explain("formatted")

In [ ]:
# Predicate pushdown with join — Catalyst automatically pushes filter before join
# Even though you write filter AFTER join, Catalyst moves it BEFORE

q_late = orders_pq.join(products_pq, on="product_id", how="inner") \
                   .filter(F.col("category") == "Electronics")

q_early = orders_pq.join(
    products_pq.filter(F.col("category") == "Electronics"),
    on="product_id", how="inner"
)

print("Filter AFTER join (Catalyst will push it down anyway):")
q_late.explain()

print("\nFilter BEFORE join (explicit):")
q_early.explain()

print("""
Key insight: both plans are IDENTICAL!
Catalyst automatically pushes the filter before the join.
This is predicate pushdown in action.
""")

In [ ]:
# Column Pruning — only read what you select
q_pruned = orders_pq.select("order_id", "amount")

print("Column pruning — only reading 2 columns from Parquet:")
q_pruned.explain("formatted")
print("Look for 'ReadSchema' in the FileScan — it only lists the 2 columns we need.")

## Part 3: Adaptive Query Execution (AQE)

In [ ]:
# Check AQE status
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"AQE coalesce enabled: {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')}")

# AQE Feature 1: Auto-coalescing shuffle partitions
# Before AQE: groupBy always creates spark.sql.shuffle.partitions partitions (default 200)
# With AQE: if data is small, Spark automatically coalesces to fewer partitions

spark.conf.set("spark.sql.shuffle.partitions", "50")  # high setting

agg_df = orders.groupBy("customer_id").agg(F.sum("amount"))
agg_df.count()  # trigger execution so AQE can observe actual partition sizes

print(f"\nAfter groupBy with 50 shuffle.partitions:")
print(f"  Actual partitions after AQE coalescing: {agg_df.rdd.getNumPartitions()}")
print("AQE collapsed many small partitions into fewer larger ones")

spark.conf.set("spark.sql.shuffle.partitions", "4")  # reset

In [ ]:
# AQE Feature 2: Runtime broadcast join conversion
# Disable auto-broadcast to show AQE overriding it
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # disable

q_aqe = orders.join(products, on="product_id", how="inner")
print("Plan WITHOUT AQE runtime stats (shows SortMergeJoin because broadcast disabled):")
q_aqe.explain()

# Execute and see what AQE actually did
q_aqe.count()
print("\nAQE may convert to BroadcastHashJoin at runtime if products fits in memory")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # reset

## Part 4: Optimization Tips — What to Look For in explain()

In [ ]:
# GOOD signs in explain():
print("""
GOOD signs in explain() output:
──────────────────────────────────────────────────────────
✓ BroadcastHashJoin         → small table broadcast (no shuffle)
✓ PushedFilters             → filter pushed to data source
✓ ReadSchema: (col1, col2)  → only needed columns read (column pruning)
✓ HashAggregate (partial)   → two-phase aggregation (efficient)
✓ AdaptiveSparkPlan         → AQE active

BAD signs in explain() output:
──────────────────────────────────────────────────────────
✗ SortMergeJoin             → both sides shuffled (expensive — can you broadcast?)
✗ CartesianProduct          → cross join (dangerous)
✗ Exchange (many times)     → too many shuffles (can you repartition less?)
✗ UDFToGenerator            → Python UDF (consider Pandas UDF or built-in)
✗ No PushedFilters          → filter not pushed down (check file format)
""")

In [ ]:
# Demonstrate: complex query — find the plan
from pyspark.sql.functions import broadcast

# Bad version: filter after joining, no broadcast
bad_query = (
    orders.join(products, on="product_id", how="inner")
          .join(customers, on="customer_id", how="inner")
          .filter(F.col("category") == "Electronics")
          .filter(F.col("country") == "India")
)

# Good version: filter early + broadcast small tables
good_query = (
    orders
    .join(broadcast(products.filter(F.col("category") == "Electronics")),
          on="product_id", how="inner")
    .join(broadcast(customers.filter(F.col("country") == "India")),
          on="customer_id", how="inner")
)

print("Bad query plan:")
bad_query.explain()

print("\nGood query plan (broadcast + early filter):")
good_query.explain()

## Part 5: Spark UI for Query Plans

In [ ]:
# Trigger a query so it shows up in the Spark UI
result = orders.join(broadcast(products), on="product_id") \
               .groupBy("category") \
               .agg(F.sum("amount").alias("revenue")) \
               .orderBy(F.col("revenue").desc())

result.show()

print("""
Now go to Spark UI → http://localhost:4040 → SQL / DataFrame tab
Click on the most recent query to see:
  - The visual query plan (boxes and arrows)
  - Each node's input/output rows
  - Time spent in each stage
  - Whether broadcast was used
  - Number of rows after each filter (to verify predicate pushdown)
""")

## Exercises

1. Write a query that joins orders + products + customers. Run `explain('formatted')` and identify: where is the data read, where is the join, and is there a broadcast?
2. Write the same query two ways: with filters applied before the join vs after. Compare the explain plans. Are they different?
3. Enable AQE (`spark.sql.adaptive.enabled = true`), then run a groupBy with `shuffle.partitions=200`. How many partitions does the result actually have?
4. What does `PushedFilters: [IsNotNull(amount), GreaterThan(amount,500.0)]` mean in a Parquet scan explain output?
5. Look up the Spark SQL tab in the Web UI (localhost:4040) after running a query. What information does the visual plan show that `explain()` doesn't?

In [ ]:
# Exercise 2: Filter before vs after join
# Write a function to compare plans

# Filter AFTER join
q1 = orders_pq.join(products_pq, on="product_id", how="inner") \
               .filter(F.col("price") > 100) \
               .filter(F.col("amount") > 200)

# Filter BEFORE join
q2 = orders_pq.filter(F.col("amount") > 200) \
               .join(products_pq.filter(F.col("price") > 100),
                    on="product_id", how="inner")

print("Q1 (filter after join) plan:")
q1.explain()
print("\nQ2 (filter before join) plan:")
q2.explain()
print("\nObservation: Catalyst pushes filters down in Q1 — same physical plan as Q2!")